In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 📈 PDI Economic Forecasting

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fpopulation_dynamics_insights%2Fnotebooks%2Ftime_series_forecasting%2Fpdi_economic_forecasting_example.ipynb?utm_source=pdi_notebooks">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/googlemaps-samples/insights-samples/blob/main/population_dynamics_insights/notebooks/time_series_forecasting/pdi_economic_forecasting_example.ipynb&utm_source=pdi_notebooks">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/googlemaps-samples/insights-samples/blob/main/population_dynamics_insights/notebooks/time_series_forecasting/pdi_economic_forecasting_example.ipynb">
      <img width="32px" src="https://upload.wikimedia.org/wikipedia/commons/9/91/Octicons-mark-github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>
<br><br><br>

> **⚠️ Important Requirement:** To run the queries in this notebook, your Google Cloud Project must have access to the **US Population Dynamics Insights dataset**. For instructions on how to request and configure access, see [Set up Population Dynamics Insights](https://developers.google.com/maps/documentation/population-dynamics-insights/cloud-setup).

### Overall Goal

This guide demonstrates how to use **Population Dynamics Insights (PDI)** to improve time-series forecasting.

**The Scenario:** You want to predict county-level unemployment rates. Traditional auto-regressive models rely entirely on historical momentum. This notebook demonstrates how to train a machine learning model using PDI's 330-dimensional embeddings to teach the model the underlying geographic, environmental, and behavioral "DNA" of a location.

By comparing a standard baseline model against a PDI-enhanced model, we mathematically isolate and visualize the predictive power of spatial features.

### Key Technologies Used

*   **[Population Dynamics Insights](https://developers.google.com/maps/documentation/population-dynamics-insights/overview):** To provide the underlying 330-dimensional embeddings capturing geographic, environmental, and map features.
*   **[BigQuery GIS](https://cloud.google.com/bigquery):** To execute spatial overlays and demographic math natively in the data warehouse.
*   **[Data Commons API](https://docs.datacommons.org/api/python/v2/):** To programmatically fetch historical economic ground-truth data.
*   **[LightGBM](https://lightgbm.readthedocs.io/):** To train gradient-boosted decision trees on tabular spatial data.

### How to Use This Notebook

1.  **Prerequisites:** Enable the **BigQuery API** and the **Google Earth Engine API** in your Google Cloud Project.
2.  **Authentication:** Configure an environment variable in the Colab "Secrets" tab named `GCP_PROJECT_ID`. A Data Commons key must also be added to the relevant secret named `DC_API_KEY`. For more information about obtaining a Data Commons key, including how to use a trial key, see the [Authentication Section](https://docs.datacommons.org/api/python/v2/#authentication) of the [Data Commons Python API V2 documentation](https://docs.datacommons.org/api/python/v2/).
3.  **Run the Cells:** Execute the cells in order from top to bottom.

In [ ]:
# Install the Data Commons client and LightGBM
!pip install "datacommons-client[Pandas]" db-dtypes lightgbm --upgrade -q

In [ ]:
# @title Step 1. Setup & Authentication
# @markdown Authenticate to Google Cloud, retrieve secrets, and initialize the BigQuery client.
import sys
import pandas as pd
import numpy as np
import warnings
import lightgbm as lgbm
from sklearn import metrics
import matplotlib.pyplot as plt
import seaborn as sns
import os
import getpass

from google.cloud import bigquery
from datacommons_client import DataCommonsClient
import google.auth

warnings.filterwarnings('ignore')

# Custom exception to halt notebook execution cleanly without a traceback
class StopExecution(BaseException):
    def _render_traceback_(self):
        return []

try:
    # ATTEMPT 1: Standard Consumer Colab
    from google.colab import auth, userdata

    GCP_PROJECT_ID = userdata.get('GCP_PROJECT_ID').strip()
    print(f"✅ Secrets retrieved for project: {GCP_PROJECT_ID}")

    DC_API_KEY = userdata.get('DC_API_KEY').strip()
    print("✅ Data Commons API Key retrieved from Colab Secrets.")

    print("🔄 Authenticating user...")
    auth.authenticate_user(project_id=GCP_PROJECT_ID)
    print("✅ User Authenticated.")

    client = bigquery.Client(project=GCP_PROJECT_ID)

except (ImportError, Exception) as e:
    # ATTEMPT 2: Colab Enterprise / Local Jupyter / Missing Secret Fallback
    print(f"ℹ️ Colab Secrets not found. Falling back to Enterprise/Local Auth...")

    # Retrieve environment credentials automatically
    credentials, GCP_PROJECT_ID = google.auth.default()
    print(f"✅ Authenticated via default credentials to Project: {GCP_PROJECT_ID}")

    # Instruct the user and securely prompt for the API key
    print("\n❌ A Data Commons key needs to be added to the relevant secret in Colab.")
    print("For more information about obtaining a Data Commons key, including how to use a trial key, see the Authentication Section:")
    print("https://docs.datacommons.org/api/python/v2/#authentication")
    print("\n🔑 Please paste your Data Commons API Key and press Enter:")

    DC_API_KEY = getpass.getpass("DC API Key: ").strip()

    if not DC_API_KEY:
        print("❌ ERROR: No Data Commons API Key provided. Stopping execution.")
        raise StopExecution()
    else:
        print("✅ Data Commons API Key securely captured.")

    # Initialize Client with enterprise credentials
    client = bigquery.Client(credentials=credentials, project=GCP_PROJECT_ID)

dc_client = DataCommonsClient(api_key=DC_API_KEY)
print("✅ BigQuery & Data Commons Clients Initialized.")

### Step 2: Extract and Aggregate PDI Data

This step retrieves the Population Dynamics Insights embeddings for counties on the US West Coast (California, Oregon, Washington).

To ensure precise spatial representation, the query performs a population-weighted aggregation. It intersects the native S2 Level 12 grid cells with public county boundaries and weights the 330-dimensional features based on high-resolution demographic data from Google Earth Engine.

> **Note:** We utilize the Python BigQuery client (`client.query`) rather than notebook magic commands to prevent session timeouts during long-running spatial queries.

In [ ]:
sql_query = """
WITH Clean_Boundaries AS (
  SELECT
    b.geo_id AS county_fips,
    s.state_name AS state_name,
    b.county_name AS county_name,
    LOWER(s.state_name) AS join_state,
    LOWER(b.county_name) AS join_county
  FROM `bigquery-public-data.geo_us_boundaries.counties` b
  JOIN `bigquery-public-data.geo_us_boundaries.states` s
    ON b.state_fips_code = s.state_fips_code
),
Base_PDI AS (
  SELECT
    features,
    `carto-os.carto.S2_BOUNDARY`(`carto-os.carto.S2_FROMTOKEN`(geo_id)) AS s2_geom,
    LOWER(administrative_area_level_1_name) AS join_state,
    REGEXP_REPLACE(LOWER(administrative_area_level_2_name), r'\s+(county|parish|city and county|municipality|census area|borough)$', '') AS join_county
  FROM `population_dynamics___us.v1_s2_z12`
  WHERE administrative_area_level_2_name IS NOT NULL
    AND administrative_area_level_1_name IN ('California', 'Oregon', 'Washington')
),
Mapped_S2 AS (
  SELECT
    p.features, p.s2_geom, c.county_fips, c.state_name, c.county_name
  FROM Base_PDI p
  JOIN Clean_Boundaries c
    ON p.join_state = c.join_state AND p.join_county = c.join_county
),
S2_Populations AS (
  SELECT
    county_fips, state_name, county_name, features,
    ST_REGIONSTATS(s2_geom, 'ee://projects/sat-io/open-datasets/WORLDPOP/pop/USA_POP_2025_CN_100M_R2025A_V1').sum AS s2_pop
  FROM Mapped_S2
),
S2_Weights AS (
  SELECT
    county_fips, state_name, county_name, features,
    s2_pop / NULLIF(SUM(s2_pop) OVER(PARTITION BY county_fips), 0) AS weight
  FROM S2_Populations WHERE s2_pop > 0
),
Weighted_Features AS (
  SELECT
    county_fips, state_name, county_name, dimension_idx,
    SUM(feature_val * weight) AS weighted_val
  FROM S2_Weights
  CROSS JOIN UNNEST(features) AS feature_val WITH OFFSET AS dimension_idx
  GROUP BY 1, 2, 3, 4
)
SELECT
  CONCAT('geoId/', county_fips) AS place_name,
  state_name,
  county_name,
  ARRAY_AGG(weighted_val ORDER BY dimension_idx) AS features
FROM Weighted_Features
GROUP BY 1, 2, 3
"""

print("⏳ Submitting regional spatial aggregation to BigQuery...")

query_job = client.query(sql_query)
county_pdi_df = query_job.to_dataframe()

print(f"✅ Query complete. Regional county features shape: {county_pdi_df.shape}")
display(county_pdi_df.head(3))

### Step 3: Fetch Ground Truth Data from Data Commons

With the geographic features aggregated, we gather the target variable data. This block connects to the Data Commons API to fetch historical `UnemploymentRate_Person` observations for the identified counties.

The retrieved data is formatted into a time-series pivot table, aligning the spatial embeddings with their chronological economic outcomes.

In [ ]:
# Flatten the PDI Embeddings into independent columns
county_raw_embeddings = county_pdi_df.set_index('place_name')
expanded_features = pd.DataFrame(county_raw_embeddings['features'].tolist(), index=county_raw_embeddings.index)
expanded_features.columns = [f'feature{i}' for i in range(330)]
county_embeddings = pd.concat([county_raw_embeddings[['state_name', 'county_name']], expanded_features], axis=1)

# Fetch Data Commons Labels in Batches
label = 'UnemploymentRate_Person'
counties = list(county_embeddings.index)
batch_size = 300
all_labels = []

print(f"Fetching Data Commons historical data for {len(counties)} regional counties...")
for start in range(0, len(counties), batch_size):
    batch = counties[start : start + batch_size]
    df_batch = dc_client.observations_dataframe(variable_dcids=[label], date="all", entity_dcids=batch)
    if df_batch is not None and not df_batch.empty:
        all_labels.append(df_batch)

df_v2 = pd.concat(all_labels)

# Format into a wide Pivot Table (Rows: Counties, Columns: Dates)
df_history = df_v2[['entity', 'date', 'value']].rename(columns={'entity': 'place_name', 'value': 'unemployment_rate'})
df_history = df_history[df_history['date'].str.contains('-')] # Isolate strictly monthly data
df_history['date'] = pd.to_datetime(df_history['date'], format='mixed')

master_pivot = df_history.pivot(index='place_name', columns='date', values='unemployment_rate')
master_pivot = master_pivot.loc[master_pivot.index.intersection(county_embeddings.index)]

# Isolate the 330 feature columns for modeling
pdi_features = county_embeddings.loc[master_pivot.index, [f'feature{i}' for i in range(330)]]

print(f"✅ Data prepared. Pivot Table Shape: {master_pivot.shape}")

### Step 4: Train "One-Shot" Machine Learning Models

To explicitly measure the predictive value of the PDI embeddings, we train two distinct LightGBM regressor models on a single cross-sectional snapshot of history.

1.  **Baseline Model:** Predicts next month's unemployment based solely on the current month's unemployment rate.
2.  **PDI-Enhanced Model:** Predicts next month's unemployment using the current month's rate alongside the 330 static geographic dimensions.

Once trained on a single month, the models are frozen and used to perform 1-step-ahead rolling predictions across a 24-month future horizon.

In [ ]:
# Initialize LightGBM hyperparameters for tabular geographic data
lgbm_params = {
    'max_leaf_nodes': 19,
    'min_child_samples': 5,
    'learning_rate': 0.05,
    'n_estimators': 400,
    'colsample_bytree': 0.8,
    'verbose': -1
}

train_X_month = pd.to_datetime('2022-05-01')
train_y_month = pd.to_datetime('2022-06-01')
print(f"🧠 Training models on Snapshot: {train_X_month.strftime('%Y-%m')} predicting {train_y_month.strftime('%Y-%m')}")

# Remove records with missing values to ensure clean training
valid_idx = master_pivot[[train_X_month, train_y_month]].dropna().index
y_train = master_pivot.loc[valid_idx, train_y_month]

# Model A: Auto-Regressive Baseline
X_train_base = master_pivot.loc[valid_idx, [train_X_month]].rename(columns={train_X_month: 'current_unemployment'})
base_model = lgbm.LGBMRegressor(**lgbm_params)
base_model.fit(X_train_base, y_train)

# Model B: PDI-Enhanced Spatial Prior
X_train_pdi = pdi_features.loc[valid_idx].copy()
X_train_pdi['current_unemployment'] = X_train_base['current_unemployment']
pdi_model = lgbm.LGBMRegressor(**lgbm_params)
pdi_model.fit(X_train_pdi, y_train)
print("✅ Both models trained.")

print("🔮 Generating rolling predictions for 2022-07 to 2024-06...")
test_months = [col for col in master_pivot.columns if pd.to_datetime('2022-06-01') <= col < pd.to_datetime('2024-07-01')]

base_predictions = pd.DataFrame(index=master_pivot.index)
pdi_predictions = pd.DataFrame(index=master_pivot.index)

# Perform rolling 1-step-ahead forecasts
for i in range(len(test_months) - 1):
    current_month = test_months[i]
    next_month = test_months[i+1]

    X_test_base = master_pivot[[current_month]].rename(columns={current_month: 'current_unemployment'})
    base_predictions[next_month] = base_model.predict(X_test_base)

    X_test_pdi = pdi_features.copy()
    X_test_pdi['current_unemployment'] = X_test_base['current_unemployment']
    pdi_predictions[next_month] = pdi_model.predict(X_test_pdi)

print("✅ Forecasting complete.")

### Step 5: Evaluate and Visualize Results

The final step evaluates the forecasting accuracy of both architectures. Standard regression metrics (MAE, MAPE, and R²) are calculated for each predicted month.

The resulting charts visualize how introducing a strong spatial prior via the PDI embeddings impacts the stability and accuracy of the economic forecast over the 24-month horizon.

In [ ]:
def evaluate(y_true, y_pred):
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    return {
        'MAE': round(metrics.mean_absolute_error(y_true[mask], y_pred[mask]), 4),
        'MAPE': round(metrics.mean_absolute_percentage_error(y_true[mask], y_pred[mask]), 4),
        'R2': round(metrics.r2_score(y_true[mask], y_pred[mask]), 4),
    }

all_metrics = []
evaluation_months = base_predictions.columns

for timestamp in evaluation_months:
    gt = master_pivot[timestamp]

    base_metrics = evaluate(gt, base_predictions[timestamp])
    base_metrics['model'] = 'Standard Auto-Regressive Baseline'
    base_metrics['step'] = timestamp
    all_metrics.append(base_metrics)

    pdi_metrics = evaluate(gt, pdi_predictions[timestamp])
    pdi_metrics['model'] = 'PDI-Enhanced Model'
    pdi_metrics['step'] = timestamp
    all_metrics.append(pdi_metrics)

all_metrics_df = pd.DataFrame(all_metrics)

print("\nMean Metrics Comparison:")
summary_table = all_metrics_df.groupby('model')[['MAE', 'MAPE', 'R2']].mean()
display(summary_table)

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(3, 1, figsize=(11, 8), sharex=True)

sns.lineplot(data=all_metrics_df, x='step', y='MAPE', hue='model', ax=ax[0], marker='o', legend=False)
ax[0].set(ylabel='MAPE')

sns.lineplot(data=all_metrics_df, x='step', y='MAE', hue='model', ax=ax[1], marker='o', legend=False)
ax[1].set(ylabel='MAE')

sns.lineplot(data=all_metrics_df, x='step', y='R2', hue='model', ax=ax[2], marker='o')
ax[2].set(ylabel='$R^2$')
ax[2].legend(title='Model Architecture', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.suptitle('Isolating the Value of Population Dynamics (Regional Overview)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Step 6: Overlaying Predictions vs. Reality

While error metrics (like MAE and R²) quantify performance, plotting the predicted curves directly over the actual ground truth provides a clear visual validation.

This step generates two charts:
1.  **Macro View:** The regional average across all counties, illustrating how well the models capture overall macroeconomic trends.
2.  **Micro View:** A specific county example, highlighting how the models predict localized economic resilience.

In [ ]:
# Calculate the mean unemployment rate across all counties for the macro view
actual_macro_trend = master_pivot[evaluation_months].mean()
pdi_macro_trend = pdi_predictions[evaluation_months].mean()
base_macro_trend = base_predictions[evaluation_months].mean()

# Select a specific representative county for the micro view (e.g., the first one in the list)
sample_county_id = master_pivot.index[0]
county_name = county_embeddings.loc[sample_county_id, 'county_name']
state_name = county_embeddings.loc[sample_county_id, 'state_name']

actual_micro_trend = master_pivot.loc[sample_county_id, evaluation_months]
pdi_micro_trend = pdi_predictions.loc[sample_county_id, evaluation_months]
base_micro_trend = base_predictions.loc[sample_county_id, evaluation_months]

# Generate the overlay charts
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

# Graph A: Regional Macro Average
axes[0].plot(evaluation_months, actual_macro_trend, label='Actual Reality (Ground Truth)', color='black', linewidth=3, marker='o')
axes[0].plot(evaluation_months, pdi_macro_trend, label='PDI-Enhanced Prediction', color='#dd8452', linewidth=2, linestyle='--', marker='s')
axes[0].plot(evaluation_months, base_macro_trend, label='Baseline Prediction', color='#4c72b0', linewidth=2, linestyle=':', marker='^')

axes[0].set_title('Macro View: Average Unemployment Rate Over Time', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Unemployment Rate (%)')
# 💡 FIX: Pushed the legend outside the right border of the top chart
axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left')

# Graph B: Single County Micro View
axes[1].plot(evaluation_months, actual_micro_trend, label='Actual Reality (Ground Truth)', color='black', linewidth=3, marker='o')
axes[1].plot(evaluation_months, pdi_micro_trend, label='PDI-Enhanced Prediction', color='#dd8452', linewidth=2, linestyle='--', marker='s')
axes[1].plot(evaluation_months, base_micro_trend, label='Baseline Prediction', color='#4c72b0', linewidth=2, linestyle=':', marker='^')

axes[1].set_title(f'Micro View: {county_name}, {state_name}', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Unemployment Rate (%)')
axes[1].set_xlabel('Date')
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()